# LLM Judging Runner

Use this notebook to run judging in two modes:
- `single_row`: fast sanity check on one row
- `full_df`: run judging on the full dataframe

In [5]:
import os
import pandas as pd

from llm_client import get_client
from llm_judge import run_judge

In [ ]:
import os
os.environ["NEBIUS_API_KEY"] = "Your_key"

In [3]:
# Optional: set key inside notebook if needed
# os.environ['NEBIUS_API_KEY'] = 'YOUR_KEY_HERE'

MODE = 'full_df'  # options: 'single_row' or 'full_df'
ROW_INDEX = 26

INPUT_EXCEL_PATH = 'ecommerce_sheet_new.xlsx'
OUTPUT_EXCEL_FULL = 'ecommerce_sheet_new_judged_full.xlsx'
OUTPUT_EXCEL_SINGLE = f'ecommerce_sheet_new_judged_row_{ROW_INDEX}.xlsx'

In [4]:
results_df = pd.read_excel(INPUT_EXCEL_PATH)
client = get_client()

if MODE == 'full_df':
    judged_df, judged_full_df = run_judge(
        results_df=results_df.copy(),
        client=client,
        excel_path=OUTPUT_EXCEL_FULL,
    )
    print(f'Judged full dataframe with {len(judged_df)} rows -> {OUTPUT_EXCEL_FULL}')

elif MODE == 'single_row':
    if not (0 <= ROW_INDEX < len(results_df)):
        raise IndexError(f'ROW_INDEX={ROW_INDEX} is out of range for dataframe size {len(results_df)}')

    single_row_df = results_df.iloc[[ROW_INDEX]].copy()
    judged_df, judged_full_df = run_judge(
        results_df=single_row_df,
        client=client,
        excel_path=OUTPUT_EXCEL_SINGLE,
    )
    print(f'Judged one row (ROW_INDEX={ROW_INDEX}) -> {OUTPUT_EXCEL_SINGLE}')

else:
    raise ValueError("MODE must be either 'single_row' or 'full_df'")

Task 5: Judging:   0%|          | 0/50 [00:00<?, ?it/s]

Judged full dataframe with 50 rows -> ecommerce_sheet_new_judged_full.xlsx


In [5]:
judged_full_df

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,length,grounding,latency_rating,cost_rating,final_score,fluency_explanation,grammar_explanation,tone_explanation,length_explanation,grounding_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy lightning-fast performance and a smooth,...",4888,462,89,good,good,...,good,good,ok,good,pass,"The description reads naturally, with a clear ...",The description is free of spelling and punctu...,"The tone is generally good, as it addresses th...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Imagine capturing life's moments in stunning d...,896,462,100,good,good,...,good,good,good,good,pass,"The description reads naturally, with a clear ...",The description is free of spelling and punctu...,"The tone is conversational and engaging, with ...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Enjoy stunning photos and effortless editing w...,568,459,65,ok,good,...,good,bad,good,good,fail,"The description reads naturally, but there's a...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 56 words in the description.,Step A: Extract required facts from SOURCE DAT...
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Imagine tuning out the world and fully immersi...,813,459,98,ok,good,...,ok,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A: required_facts = ['active noise cancel...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Enjoy your audio in total harmony with the Bos...,763,448,90,ok,good,...,ok,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
5,Amazon Echo Dot (5th Gen),"features: Alexa voice assistant, temperature s...",recycled fabric mesh,1‑year limited warranty,Enjoy seamless control over your smart home wi...,718,448,80,ok,good,...,good,good,good,good,pass,"The description reads naturally, but there are...",There are no spelling or punctuation errors in...,The description uses some corporate filler phr...,I counted 69 words in the description.,Step A: Extract required facts from SOURCE DAT...
6,Dell XPS 13 9310 Laptop,"features: 13.4″ FHD+ display, Intel Core i5, T...",CNC aluminum & carbon‑fiber palmrest,1‑year limited hardware warranty,Enjoy effortless productivity with the Dell XP...,844,468,97,good,good,...,ok,good,good,good,pass,"The description reads naturally, with a clear ...",The description has zero spelling or punctuati...,"The tone is professional and informative, addr...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
7,Apple MacBook Air 13″ (M3),"features: M3 chip, 18‑hour battery, 1080p came...",recycled aluminum unibody,1‑year limited warranty,Imagine enjoying 18 hours of work or play on a...,1009,462,100,ok,good,...,good,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it add

In [6]:
judged_full_df['grounding_explanation'].values

array(["Step A: Extract required facts from SOURCE DATA. required_facts = ['features: Light BOOST midsole, Primeknit+ upper, Continental rubber', 'rating: 4.7/5', 'material: textile & synthetic', 'warranty: 2‑year manufacturer warranty']. rating_required = true. required_rating_value = '4.7/5'. Step B: Extract claimed facts from DESCRIPTION only. claimed_facts = ['responsive Light BOOST midsole', 'breathable Primeknit+ upper', 'reliable grip', '2-year warranty']. Step C: Coverage/support checks. For each required_fact, mark COVERED or MISSING in DESCRIPTION. For each claimed_fact, mark SUPPORTED or UNSUPPORTED vs SOURCE. rating_covered = true. Step D: Mandatory decision gates. Since all claimed_facts are SUPPORTED and all required_facts are COVERED, and rating_covered is true, the description passes the mandatory decision gates. Step E: If no mandatory bad gate triggered, the description is good if all claimed_facts are SUPPORTED and all required_facts are COVERED. Since this is the ca

In [6]:
judged_full_df.to_excel(f'ecommerce_sheet_new_judged_full.xlsx')

In [16]:
judged_full_df.to_excel(f'ecommerce_sheet_new_judged_row_{ROW_INDEX}_full.xlsx', index=False)

In [ ]:
display(judged_df.head())
display(judged_full_df.head())

In [5]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm


# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "grounding"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")

df_separate.head()

  0%|          | 0/50 [00:00<?, ?it/s]

Saved separate metric results to: ecommerce_sheet_new_grounding_separate.xlsx


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,grounding_separate_verdict,grounding_separate_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,ok,good,bad,good,good,fail,good,Grounding evaluation for Apple iPhone 15 Pro
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,ok,good,good,good,good,pass,good,Grounding evaluation for Samsung Galaxy S24 Ultra
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,ok,good,ok,good,good,pass,good,Grounding evaluation for Google Pixel 8 Pro
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,good,ok,good,good,good,pass,good,Grounding evaluation for Sony WH-1000XM5 Headp...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,ok,ok,good,good,good,pass,good,Grounding evaluation for Bose QuietComfort Ult...


In [ ]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm

# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "tone"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"



  0%|          | 0/50 [00:00<?, ?it/s]

PermissionError: [Errno 13] Permission denied: 'ecommerce_sheet_new_tone_separate.xlsx'

In [ ]:
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")


Saved separate metric results to: ecommerce_sheet_new_tone_separate.xlsx


In [6]:
df_separate.head()

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,tone_separate_verdict,tone_separate_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,ok,good,bad,good,good,fail,ok,The description uses buzz words ('seamless') a...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,ok,good,good,good,good,pass,ok,The description uses buzz words ('incredible')...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,ok,good,ok,good,good,pass,good,"The description uses a conversational tone, ad..."
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,good,ok,good,good,good,pass,ok,The description addresses the customer directl...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,ok,ok,good,good,good,pass,ok,The description uses a benefit-focused opening...


In [4]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm

# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "fluency"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"



  0%|          | 0/50 [00:00<?, ?it/s]

In [5]:
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")


Saved separate metric results to: ecommerce_sheet_new_fluency_separate.xlsx


In [6]:
df_separate.head()

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,fluency_separate_verdict,fluency_separate_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,ok,good,bad,good,good,fail,good,The description reads naturally and flows well...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,ok,good,good,good,good,pass,good,The description reads naturally and flows well...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,ok,good,ok,good,good,pass,good,The description reads naturally and flows well...
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,good,ok,good,good,good,pass,good,"The description flows naturally and logically,..."
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,ok,ok,good,good,good,pass,good,The description reads naturally and flows well...


In [7]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm

# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "grammar"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")



  0%|          | 0/50 [00:00<?, ?it/s]

Saved separate metric results to: ecommerce_sheet_new_grammar_separate.xlsx


## Comparing multi metric prompt to metric specific prompt:

In [4]:
full_df_path = 'assignment_01_full_new.xlsx'
full_df = pd.read_excel(full_df_path)